<a href="https://colab.research.google.com/github/jac0bmath3w/rail-safety-ai/blob/main/notebooks/01-ingest_embed_store.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/jac0bmath3w/rail-safety-ai.git

In [ ]:
!pip install pypdf
!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install llama-index
!pip install -U langchain-text-splitters
!pip install chromadb

In [ ]:
import sys, os
import importlib
# sys.path.append('/content/rail-safety-ai/src')
src_path = os.path.join(os.getcwd(), 'rail-safety-ai', 'src')
sys.path.append(src_path)
raw_data_path = os.path.join(os.getcwd(), 'rail-safety-ai', 'data', 'raw')


from ingest import RailDocumentProcessor
from embed import RailEmbedder
from vector_store import RailVectorVault


In [ ]:

processor = RailDocumentProcessor()
all_chunks, all_metadata = processor.process_directory(raw_data_path)

print(f"Total Knowledge Base Chunks: {len(all_chunks)}")

In [ ]:
embedder = RailEmbedder(model_name = 'BAAI/bge-base-en-v1.5')
# vectors = embedder.generate_embeddings(all_chunks)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
vault = RailVectorVault(
    embedder_instance=embedder,
    db_path="/content/drive/My Drive/rail_ai_project/vector_db"
)

In [ ]:
vault.add_documents(all_chunks, all_metadata)

In [ ]:
q_vec = vault.embedder.model.encode('When can a crossing be considered for grade separation').tolist()

In [ ]:
result = vault.collection.query(query_embeddings=q_vec, n_results=5)

In [ ]:
result

In [ ]:
from engine import RailSafetyEngine
engine = RailSafetyEngine()

In [ ]:
chunks = result['documents'][0]

In [ ]:
question = "When can a crossing be considered for grade separation?"
answer = engine.generate_answer(question, chunks)

In [ ]:
print(answer)

In [ ]:
DB_PATH = "/content/drive/My Drive/rail_ai_project/vector_db"
v2 = RailVectorVault(embedder_instance=embedder, db_path=DB_PATH)

In [ ]:
print(v2.collection.count())

In [ ]:
all_data = v2.collection.get(include = ['documents', 'metadatas', 'embeddings'])

In [ ]:
import pandas as pd
ids = all_data['ids']
documents = all_data['documents']
metadatas = all_data['metadatas']

# 3. Create a DataFrame
# This automatically turns the list of metadata dicts into separate columns
df = pd.DataFrame(metadatas)

# 4. Add the IDs and Text Chunks as the first columns
df.insert(0, 'chunk_id', ids)
df.insert(1, 'text_content', documents)


In [ ]:
df.to_csv('/content/drive/MyDrive/chunks.csv', index = False)